In [6]:
# ============================================================
# PATCH: COMPATIBLE PLAIN TEXT LIVE PROGRESS BAR
# Fixes: TypeError: 'TextProgressUI' object is not subscriptable
# This works with the existing run_pipeline() code.
# ============================================================

import time
import html
from IPython.display import display, update_display, HTML

class _ProgressProxy:
    def __init__(self, ui, name):
        object.__setattr__(self, "ui", ui)
        object.__setattr__(self, "name", name)

    @property
    def value(self):
        if self.name == "epoch_bar":
            return self.ui.epoch
        if self.name == "batch_bar":
            return self.ui.current
        return 0

    @value.setter
    def value(self, v):
        try:
            v = int(v)
        except Exception:
            v = 0

        if self.name == "epoch_bar":
            self.ui.epoch = v
        elif self.name == "batch_bar":
            self.ui.current = v

        self.ui.render()

    @property
    def max(self):
        if self.name == "epoch_bar":
            return self.ui.total_epochs
        if self.name == "batch_bar":
            return self.ui.total
        return 1

    @max.setter
    def max(self, v):
        try:
            v = int(v)
        except Exception:
            v = 1

        if self.name == "epoch_bar":
            self.ui.total_epochs = max(v, 1)
        elif self.name == "batch_bar":
            self.ui.total = max(v, 1)

        self.ui.render()

    @property
    def description(self):
        return self.ui.phase

    @description.setter
    def description(self, v):
        self.ui.phase = str(v)
        self.ui.render()


class TextProgressUI:
    def __init__(self, title="Official AdaGate-DF on DF40"):
        self.display_id = "adagatedf_live_text_progress"
        self.title = title

        self.epoch = 0
        self.total_epochs = globals().get("EPOCHS", 1)

        self.phase = "Preparing"
        self.current = 0
        self.total = 1

        self.start_time = None
        self.loss = None
        self.acc = None
        self.extra = ""

        self.status = "Preparing..."
        self.metrics = ""
        self.files = ""

        self._proxies = {
            "epoch_bar": _ProgressProxy(self, "epoch_bar"),
            "batch_bar": _ProgressProxy(self, "batch_bar"),
        }

        display(HTML(self._html()), display_id=self.display_id)

    def __getitem__(self, key):
        if key in self._proxies:
            return self._proxies[key]
        raise KeyError(key)

    def _bar(self, current, total, width=28):
        total = max(int(total), 1)
        current = max(0, min(int(current), total))
        frac = current / total
        filled = int(width * frac)
        empty = width - filled
        return f"{frac * 100:5.1f}%|" + "█" * filled + "░" * empty + f"| {current}/{total}"

    def _format_time(self, seconds):
        seconds = max(0, int(seconds))
        h = seconds // 3600
        m = (seconds % 3600) // 60
        s = seconds % 60
        if h > 0:
            return f"{h}:{m:02d}:{s:02d}"
        return f"{m}:{s:02d}"

    def _html(self):
        line = (
            f"{self.phase} epoch {self.epoch}/{self.total_epochs}: "
            f"{self._bar(self.current, self.total)}"
        )

        if self.start_time is not None and self.current > 0:
            elapsed = time.time() - self.start_time
            sec_per_it = elapsed / max(self.current, 1)
            eta = sec_per_it * max(self.total - self.current, 0)
            line += f" [{self._format_time(elapsed)}<{self._format_time(eta)}, {sec_per_it:.2f}s/it]"

        metric_bits = []
        if self.loss is not None:
            metric_bits.append(f"loss={self.loss:.4f}")
        if self.acc is not None:
            metric_bits.append(f"acc={self.acc:.4f}")
        if self.extra:
            metric_bits.append(str(self.extra))

        if metric_bits:
            line += ", " + ", ".join(metric_bits)

        parts = [
            f"<b>{html.escape(self.title)}</b>",
            "",
            html.escape(line),
            "",
            html.escape(str(self.status)),
        ]

        if self.metrics:
            parts += ["", "<b>Latest metrics</b>", f"<pre>{html.escape(str(self.metrics))}</pre>"]

        if self.files:
            parts += ["", "<b>Output files</b>", f"<pre>{html.escape(str(self.files))}</pre>"]

        return (
            "<div style='font-family: monospace; white-space: pre-wrap; "
            "border:1px solid #ccc; padding:12px; border-radius:8px; "
            "font-size:15px; line-height:1.4;'>"
            + "\n".join(parts)
            + "</div>"
        )

    def render(self):
        update_display(HTML(self._html()), display_id=self.display_id)

    def update(
        self,
        phase,
        current,
        total,
        epoch=None,
        total_epochs=None,
        start_time=None,
        loss=None,
        acc=None,
        extra=None,
    ):
        self.phase = str(phase)

        self.current = int(current)
        self.total = max(int(total), 1)

        if epoch is not None:
            self.epoch = int(epoch)

        if total_epochs is not None:
            self.total_epochs = max(int(total_epochs), 1)

        if start_time is not None:
            self.start_time = start_time

        self.loss = loss
        self.acc = acc
        self.extra = "" if extra is None else str(extra)

        self.render()

    def set_stage(self, text):
        self.status = str(text)
        self.render()

    def set_metrics(self, text):
        self.metrics = str(text)
        self.render()

    def set_files(self, text):
        self.files = str(text)
        self.render()


def make_ui():
    return TextProgressUI("Official AdaGate-DF on DF40")


def update_stage(ui, text):
    ui.set_stage(text)


def update_metrics(ui, text):
    ui.set_metrics(text)


def update_files(ui, text):
    ui.set_files(text)


def train_one_epoch_live(model, loader, optimizer, scaler, device, ui, epoch_idx):
    import torch
    from torch.cuda.amp import autocast
    from deepfake_lowres.data.gated_dataset import make_gate_targets_from_quality_features
    from deepfake_lowres.models.gated_dual_branch import compute_total_loss

    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    amp_enabled = MIXED_PRECISION and device.type == "cuda"
    start_time = time.time()
    last_update = 0

    for step, batch in enumerate(loader, start=1):
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=amp_enabled):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            preds = outputs["fused_logits"].argmax(dim=1)
            correct = (preds == labels).sum().item()

        bs = labels.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        total_correct += correct
        total_count += bs

        avg_loss = total_loss / max(total_count, 1)
        avg_acc = total_correct / max(total_count, 1)

        now = time.time()
        if (
            step == 1
            or step == len(loader)
            or step % UI_UPDATE_EVERY_STEPS == 0
            or now - last_update >= UI_UPDATE_EVERY_SECONDS
        ):
            ui.update(
                phase="Train",
                current=step,
                total=len(loader),
                epoch=epoch_idx,
                total_epochs=EPOCHS,
                start_time=start_time,
                loss=avg_loss,
                acc=avg_acc,
            )
            last_update = now

    return {
        "train_loss": total_loss / max(total_count, 1),
        "train_accuracy": total_correct / max(total_count, 1),
    }


def predict_live(model, loader, device, ui, stage_name, source_df):
    import pandas as pd
    import torch
    from torch.cuda.amp import autocast

    model.eval()

    rows = []
    amp_enabled = MIXED_PRECISION and device.type == "cuda"

    path_to_method = dict(zip(source_df["image_path"].astype(str), source_df["method"].astype(str)))
    path_to_video = dict(zip(source_df["image_path"].astype(str), source_df["video_id"].astype(str)))

    start_time = time.time()
    last_update = 0

    with torch.no_grad():
        for step, batch in enumerate(loader, start=1):
            rgb = batch["rgb"].to(device, non_blocking=True)
            freq = batch["freq"].to(device, non_blocking=True)
            qf = batch["quality_features"].to(device, non_blocking=True)

            with autocast(enabled=amp_enabled):
                pred_out = model.predict(rgb, freq, qf, threshold=UNCERTAIN_THRESHOLD)

            probs = pred_out["probs"].detach().cpu().numpy()
            fake_prob = probs[:, 1]
            pred = pred_out["pred"].detach().cpu().numpy()
            routes = pred_out["routes"].detach().cpu().numpy()
            conf = pred_out["conf"].detach().cpu().numpy()
            decisions = pred_out["decision"]

            labels = batch["label"].detach().cpu().numpy()
            resolutions = batch["resolution"].detach().cpu().numpy()

            for i in range(len(fake_prob)):
                path = str(batch["path"][i])
                rows.append({
                    "image_path": path,
                    "video_id": path_to_video.get(path, path),
                    "method": path_to_method.get(path, "unknown_method"),
                    "label": int(labels[i]),
                    "fake_prob": float(fake_prob[i]),
                    "pred_label": int(pred[i]),
                    "confidence": float(conf[i]),
                    "decision": str(decisions[i]),
                    "route": int(routes[i]),
                    "resolution": int(resolutions[i]),
                })

            now = time.time()
            if (
                step == 1
                or step == len(loader)
                or step % UI_UPDATE_EVERY_STEPS == 0
                or now - last_update >= UI_UPDATE_EVERY_SECONDS
            ):
                ui.update(
                    phase=str(stage_name),
                    current=step,
                    total=len(loader),
                    epoch=getattr(ui, "epoch", 1),
                    total_epochs=EPOCHS,
                    start_time=start_time,
                    extra=f"rows={len(rows)}",
                )
                last_update = now

    elapsed = time.time() - start_time
    return pd.DataFrame(rows), elapsed


print("PASS: patched compatible live text progress UI.")
print("Now run:")
print("results = run_pipeline()")

PASS: patched compatible live text progress UI.
Now run:
results = run_pipeline()


In [3]:
# ============================================================
# SELF-CONTAINED ADAGATE RESUME + FINAL EVAL CELL
# Use this in a fresh notebook/kernel.
# ============================================================

import os
import sys
import time
import json
import math
import shutil
import zipfile
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
ZIP_PATH = Path("/mnt/data/AdaGate-DF-main.zip")
REPO_DIR = PROJECT_ROOT / "AdaGate-DF"
SRC_DIR = REPO_DIR / "src"

RAW_DF40_MANIFEST = PROJECT_ROOT / "outputs_df40/df40_manifest_all_clean.csv"
SAFE_DF40_MANIFEST = PROJECT_ROOT / "outputs_df40/dataset_summary/df40_manifest_adagate_safe.csv"

FAKE_TEST_ROOT = PROJECT_ROOT / "df40/Fake_Test"
OOD_OUT_ROOT = PROJECT_ROOT / "outputs_df40_ood_fake_test_all_models"
OOD_MANIFEST = OOD_OUT_ROOT / "df40_fake_test_manifest_adagate_official.csv"

OUT_DIR = PROJECT_ROOT / "outputs_adagate_df40_indomain"
OUT_EVAL = PROJECT_ROOT / "outputs_adagate_df40_indomain_eval"
OUT_OOD = PROJECT_ROOT / "outputs_adagate_df40_ood_fake_test"

LATEST_CKPT = OUT_DIR / "latest_gated_dual_branch_df40.pt"
BEST_CKPT = OUT_DIR / "best_gated_dual_branch_df40.pt"
HISTORY_CSV = OUT_DIR / "training_period_results.csv"

# Match your previous AdaGate run.
EPOCHS = 10
BATCH_SIZE = 16
NUM_WORKERS = 0
IMAGE_SIZE = 224
LR = 2e-4
WEIGHT_DECAY = 1e-4
MIXED_PRECISION = True
SEED = 42

TRAIN_RESOLUTIONS = [128, 224, 256, 384]
EVAL_RESOLUTIONS = [224]

DECISION_THRESHOLD = 0.5
UNCERTAIN_THRESHOLD = 0.65

FRAMES_PER_VIDEO = 32

LIMIT_TRAIN_ROWS = None
LIMIT_VAL_ROWS = None
LIMIT_TEST_ROWS = None
LIMIT_OOD_ROWS = None

BALANCE_MODE = "none"

RUN_TRAIN_RESUME = True
RUN_INDOMAIN_EVAL = True
RUN_OOD_EVAL = True

UI_UPDATE_EVERY_STEPS = 10
UI_UPDATE_EVERY_SECONDS = 0.5


# ============================================================
# BASIC HELPERS
# ============================================================

def print_header(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def require_path(path, name):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    print(f"FOUND {name}: {path}")


def prepare_repo():
    require_path(REPO_DIR, "AdaGate repo")
    require_path(RAW_DF40_MANIFEST, "DF40 clean manifest")
    require_path(FAKE_TEST_ROOT, "DF40 Fake_Test folder")

    src_root = REPO_DIR / "src"
    pkg_root = src_root / "deepfake_lowres"
    pkg_root.mkdir(parents=True, exist_ok=True)

    for name in ["data", "models", "training", "results"]:
        src = src_root / name
        dst = pkg_root / name
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)

    for name in ["metrics.py", "utils.py"]:
        src = src_root / name
        dst = pkg_root / name
        if src.exists():
            shutil.copy2(src, dst)

    for d in [
        pkg_root,
        pkg_root / "data",
        pkg_root / "models",
        pkg_root / "training",
        pkg_root / "results",
    ]:
        d.mkdir(parents=True, exist_ok=True)
        (d / "__init__.py").touch()

    if str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    print("AdaGate package ready:", pkg_root)


def seed_everything(seed):
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def check_gpu_processes():
    print_header("GPU process check")
    try:
        out = subprocess.check_output(
            ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.STDOUT,
        ).strip()
        print(out if out else "No GPU compute processes.")
    except Exception as e:
        print("Could not query nvidia-smi:", repr(e))


# ============================================================
# SIMPLE LIVE TEXT PROGRESS DISPLAY
# ============================================================

class LiveProgress:
    def __init__(self, title="AdaGate DF40"):
        from IPython.display import display, update_display, HTML
        self.display = display
        self.update_display = update_display
        self.HTML = HTML
        self.display_id = "adagatedf_resume_progress"
        self.title = title
        self.text = "Starting..."
        display(HTML(self._html()), display_id=self.display_id)

    def _bar(self, current, total, width=28):
        total = max(int(total), 1)
        current = max(0, min(int(current), total))
        frac = current / total
        filled = int(width * frac)
        return f"{frac*100:5.1f}%|" + "█" * filled + "░" * (width - filled) + f"| {current}/{total}"

    def _fmt_time(self, seconds):
        seconds = max(0, int(seconds))
        h = seconds // 3600
        m = (seconds % 3600) // 60
        s = seconds % 60
        if h:
            return f"{h}:{m:02d}:{s:02d}"
        return f"{m}:{s:02d}"

    def _html(self):
        import html
        return (
            "<div style='font-family: monospace; white-space: pre-wrap; "
            "border:1px solid #ccc; padding:12px; border-radius:8px; "
            "font-size:15px; line-height:1.4;'>"
            + html.escape(self.text)
            + "</div>"
        )

    def set_text(self, text):
        self.text = text
        self.update_display(self.HTML(self._html()), display_id=self.display_id)

    def update(self, phase, current, total, epoch=None, total_epochs=None, start_time=None, loss=None, acc=None, extra=None):
        line = f"{phase}"
        if epoch is not None and total_epochs is not None:
            line += f" epoch {epoch}/{total_epochs}"
        line += f": {self._bar(current, total)}"

        if start_time is not None and current > 0:
            elapsed = time.time() - start_time
            sec_per_it = elapsed / max(current, 1)
            eta = sec_per_it * max(total - current, 0)
            line += f" [{self._fmt_time(elapsed)}<{self._fmt_time(eta)}, {sec_per_it:.2f}s/it]"

        parts = [f"{self.title}", "", line]

        metric_bits = []
        if loss is not None:
            metric_bits.append(f"loss={loss:.4f}")
        if acc is not None:
            metric_bits.append(f"acc={acc:.4f}")
        if extra:
            metric_bits.append(str(extra))
        if metric_bits:
            parts.append(", ".join(metric_bits))

        self.set_text("\n".join(parts))


# ============================================================
# DATA HELPERS
# ============================================================

def make_safe_df40_manifest():
    import pandas as pd

    df = pd.read_csv(RAW_DF40_MANIFEST)

    path_col = None
    for c in ["frame_path", "image_path", "path", "filepath", "file_path"]:
        if c in df.columns:
            path_col = c
            break

    if path_col is None:
        raise ValueError(f"No image path column found. Columns: {list(df.columns)}")

    if "label_name" not in df.columns:
        df["label_name"] = df["label"].map({0: "Real", 1: "Fake"}).fillna(df["label"].astype(str))

    if "video_id" in df.columns:
        video_col = "video_id"
    elif "group_id" in df.columns:
        video_col = "group_id"
    else:
        df["video_id"] = df[path_col].astype(str)
        video_col = "video_id"

    safe = pd.DataFrame({
        "frame_path": df[path_col].astype(str),
        "image_path": df[path_col].astype(str),
        "split": df["split"].astype(str).str.lower(),
        "label": df["label"].astype(int),
        "label_name": df["label_name"].astype(str),
        "video_id": df[video_col].astype(str),
        "is_oversampled_real": False,
    })

    if "method" in df.columns:
        safe["method"] = df["method"].astype(str)
    elif "top_level_folder" in df.columns:
        safe["method"] = df["top_level_folder"].astype(str)
    else:
        safe["method"] = "unknown_method"

    before = len(safe)
    safe = safe[safe["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
    after = len(safe)

    SAFE_DF40_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    safe.drop(columns=["image_path"]).to_csv(SAFE_DF40_MANIFEST, index=False)

    print("Saved safe manifest:", SAFE_DF40_MANIFEST)
    print("Rows:", before, "->", after)
    print(pd.crosstab(safe["split"], safe["label"]))

    return safe


def sample_frames_per_video(df, frames_per_video, seed=42):
    import numpy as np
    import pandas as pd

    if frames_per_video is None or frames_per_video <= 0:
        return df.reset_index(drop=True)

    if "video_id" not in df.columns:
        return df.reset_index(drop=True)

    parts = []
    for _, g in df.groupby("video_id", sort=False):
        g = g.sort_values("image_path").reset_index(drop=True)
        if len(g) <= frames_per_video:
            parts.append(g)
        else:
            idx = np.linspace(0, len(g) - 1, frames_per_video).astype(int)
            parts.append(g.iloc[idx])

    if not parts:
        return df.iloc[0:0].copy()

    out = pd.concat(parts, ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def limit_df(df, limit, seed=42):
    if limit is None:
        return df.reset_index(drop=True)
    limit = int(limit)
    if limit <= 0 or len(df) <= limit:
        return df.reset_index(drop=True)
    return df.sample(n=limit, random_state=seed).reset_index(drop=True)


def balance_train_dataframe_local(train_df, mode="none", seed=42):
    import pandas as pd

    train_df = train_df.copy().reset_index(drop=True)
    train_df["is_oversampled_real"] = False

    if mode is None or mode == "none":
        return train_df

    real_df = train_df[train_df["label"] == 0].copy()
    fake_df = train_df[train_df["label"] == 1].copy()

    if len(real_df) == 0 or len(fake_df) == 0:
        raise ValueError("Both classes must be present before balancing.")

    if mode == "undersample":
        n = min(len(real_df), len(fake_df))
        out = pd.concat([
            real_df.sample(n=n, random_state=seed),
            fake_df.sample(n=n, random_state=seed),
        ])
        return out.sample(frac=1, random_state=seed).reset_index(drop=True)

    if mode == "oversample_real":
        real_os = real_df.sample(n=len(fake_df), replace=True, random_state=seed).copy()
        real_os["is_oversampled_real"] = True
        fake_df["is_oversampled_real"] = False
        return pd.concat([real_os, fake_df]).sample(frac=1, random_state=seed).reset_index(drop=True)

    if mode == "oversample_minority":
        if len(real_df) < len(fake_df):
            minority, majority = real_df, fake_df
        else:
            minority, majority = fake_df, real_df
        minority_os = minority.sample(n=len(majority), replace=True, random_state=seed).copy()
        return pd.concat([minority_os, majority]).sample(frac=1, random_state=seed).reset_index(drop=True)

    raise ValueError(f"Unknown balance mode: {mode}")


def build_splits(safe_df):
    train_df = safe_df[safe_df["split"] == "train"].reset_index(drop=True)
    val_df = safe_df[safe_df["split"].isin(["val", "valid", "validation"])].reset_index(drop=True)
    test_df = safe_df[safe_df["split"] == "test"].reset_index(drop=True)

    train_df = sample_frames_per_video(train_df, FRAMES_PER_VIDEO, SEED)
    val_df = sample_frames_per_video(val_df, FRAMES_PER_VIDEO, SEED)
    test_df = sample_frames_per_video(test_df, FRAMES_PER_VIDEO, SEED)

    train_df = limit_df(train_df, LIMIT_TRAIN_ROWS, SEED)
    val_df = limit_df(val_df, LIMIT_VAL_ROWS, SEED)
    test_df = limit_df(test_df, LIMIT_TEST_ROWS, SEED)

    train_df = balance_train_dataframe_local(train_df, BALANCE_MODE, SEED)

    print("\nSplit summary after sampling/limits:")
    for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        real = int((split_df["label"] == 0).sum())
        fake = int((split_df["label"] == 1).sum())
        print(f"{name:5s} real={real:7d} fake={fake:7d} total={len(split_df):7d}")

    return train_df, val_df, test_df


def build_ood_manifest():
    import pandas as pd
    from PIL import Image, ImageFile

    ImageFile.LOAD_TRUNCATED_IMAGES = True
    OOD_OUT_ROOT.mkdir(parents=True, exist_ok=True)

    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    paths = sorted([p for p in FAKE_TEST_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in image_exts])

    rows = []
    bad = []

    for i, p in enumerate(paths):
        try:
            if p.stat().st_size == 0:
                raise RuntimeError("empty_file")
            with Image.open(p) as img:
                img.verify()
            with Image.open(p) as img:
                img.convert("RGB")
        except Exception as e:
            bad.append({"frame_path": str(p), "reason": repr(e)})
            continue

        rel = p.relative_to(FAKE_TEST_ROOT)
        method = rel.parts[0] if len(rel.parts) > 1 else "unknown_method"
        video_id = f"{method}|{str(rel.with_suffix(''))}|{i}"

        rows.append({
            "frame_path": str(p),
            "image_path": str(p),
            "split": "test",
            "label": 1,
            "label_name": "Fake",
            "video_id": video_id,
            "method": method,
            "is_oversampled_real": False,
        })

    ood_df = pd.DataFrame(rows)
    bad_df = pd.DataFrame(bad)

    if len(ood_df) > 0:
        ood_df.drop(columns=["image_path"]).to_csv(OOD_MANIFEST, index=False)

    bad_df.to_csv(OOD_OUT_ROOT / "df40_fake_test_bad_images_adagate_official.csv", index=False)

    print("Saved OOD manifest:", OOD_MANIFEST)
    print("OOD valid images:", len(ood_df))
    print("OOD bad images:", len(bad_df))

    if LIMIT_OOD_ROWS is not None:
        ood_df = limit_df(ood_df, LIMIT_OOD_ROWS, SEED)

    return ood_df


# ============================================================
# METRICS
# ============================================================

def compute_eer(y_true, y_score):
    import numpy as np
    from sklearn.metrics import roc_curve

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if len(np.unique(y_true)) < 2:
        return float("nan")

    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1.0 - tpr
    i = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[i] + fnr[i]) / 2.0)


def tpr_at_fpr(y_true, y_score, target_fpr):
    import numpy as np
    from sklearn.metrics import roc_curve

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if len(np.unique(y_true)) < 2:
        return float("nan")

    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    idx = np.where(fpr <= target_fpr)[0]
    if len(idx) == 0:
        return 0.0
    return float(np.max(tpr[idx]))


def compute_metrics(y_true, y_score, threshold=0.5, confidence=None, uncertain_threshold=0.65):
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        average_precision_score,
        roc_auc_score,
        precision_recall_fscore_support,
    )

    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "Decision Threshold": float(threshold),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
    }

    if len(np.unique(y_true)) >= 2:
        out["Balanced Accuracy"] = float(balanced_accuracy_score(y_true, y_pred))
        out["AUPRC"] = float(average_precision_score(y_true, y_score))
        out["AUCROC"] = float(roc_auc_score(y_true, y_score))
        out["EER (Equal Error Rate)"] = compute_eer(y_true, y_score)
        out["TPR@1%FPR"] = tpr_at_fpr(y_true, y_score, 0.01)
        out["TPR@0.1%FPR"] = tpr_at_fpr(y_true, y_score, 0.001)
    else:
        out["Balanced Accuracy"] = float("nan")
        out["AUPRC"] = float("nan")
        out["AUCROC"] = float("nan")
        out["EER (Equal Error Rate)"] = float("nan")
        out["TPR@1%FPR"] = float("nan")
        out["TPR@0.1%FPR"] = float("nan")

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    out["Precision (0)"] = float(precision[0])
    out["Recall (0)"] = float(recall[0])
    out["F1 (0)"] = float(f1[0])
    out["Precision (1)"] = float(precision[1])
    out["Recall (1)"] = float(recall[1])
    out["F1 (1)"] = float(f1[1])
    out["Real Support"] = int(support[0])
    out["Fake Support"] = int(support[1])

    if confidence is None:
        confidence = np.maximum(y_score, 1.0 - y_score)

    confidence = np.asarray(confidence).astype(float)
    out["Uncertain Rate (%)"] = float((confidence < uncertain_threshold).mean() * 100.0)

    out["Predicted Real Count"] = int((y_pred == 0).sum())
    out["Predicted Fake Count"] = int((y_pred == 1).sum())

    out["Fake Detection Rate / Recall(1)"] = out["Recall (1)"]
    out["Missed Fake Rate"] = float(1.0 - out["Recall (1)"]) if out["Fake Support"] > 0 else float("nan")

    return out


# ============================================================
# TRAIN/EVAL HELPERS
# ============================================================

def make_loader(df, mode, resolutions, batch_size, shuffle):
    from torch.utils.data import DataLoader
    from deepfake_lowres.data.gated_dataset import GatedMultiResolutionDataset

    ds = GatedMultiResolutionDataset(
        df,
        image_size=IMAGE_SIZE,
        resolutions=resolutions,
        mode=mode,
        cache_features=False,
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=(mode == "train_random"),
    )

    return loader


def train_one_epoch(model, loader, optimizer, scaler, device, progress, epoch_idx):
    import torch
    from torch.cuda.amp import autocast
    from deepfake_lowres.data.gated_dataset import make_gate_targets_from_quality_features
    from deepfake_lowres.models.gated_dual_branch import compute_total_loss

    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    amp_enabled = MIXED_PRECISION and device.type == "cuda"
    start_time = time.time()
    last_update = 0

    for step, batch in enumerate(loader, start=1):
        rgb = batch["rgb"].to(device, non_blocking=True)
        freq = batch["freq"].to(device, non_blocking=True)
        qf = batch["quality_features"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        gate_targets = make_gate_targets_from_quality_features(qf).to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=amp_enabled):
            outputs = model(rgb, freq, qf)
            loss = compute_total_loss(outputs, labels, gate_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            preds = outputs["fused_logits"].argmax(dim=1)
            correct = (preds == labels).sum().item()

        bs = labels.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        total_correct += correct
        total_count += bs

        avg_loss = total_loss / max(total_count, 1)
        avg_acc = total_correct / max(total_count, 1)

        now = time.time()
        if step == 1 or step == len(loader) or step % UI_UPDATE_EVERY_STEPS == 0 or now - last_update >= UI_UPDATE_EVERY_SECONDS:
            progress.update(
                phase="Train",
                current=step,
                total=len(loader),
                epoch=epoch_idx,
                total_epochs=EPOCHS,
                start_time=start_time,
                loss=avg_loss,
                acc=avg_acc,
            )
            last_update = now

    return {
        "train_loss": total_loss / max(total_count, 1),
        "train_accuracy": total_correct / max(total_count, 1),
    }


def predict(model, loader, device, progress, stage_name, source_df):
    import pandas as pd
    import torch
    from torch.cuda.amp import autocast

    model.eval()

    rows = []
    amp_enabled = MIXED_PRECISION and device.type == "cuda"

    path_to_method = dict(zip(source_df["image_path"].astype(str), source_df["method"].astype(str)))
    path_to_video = dict(zip(source_df["image_path"].astype(str), source_df["video_id"].astype(str)))

    start_time = time.time()
    last_update = 0

    with torch.no_grad():
        for step, batch in enumerate(loader, start=1):
            rgb = batch["rgb"].to(device, non_blocking=True)
            freq = batch["freq"].to(device, non_blocking=True)
            qf = batch["quality_features"].to(device, non_blocking=True)

            with autocast(enabled=amp_enabled):
                pred_out = model.predict(rgb, freq, qf, threshold=UNCERTAIN_THRESHOLD)

            probs = pred_out["probs"].detach().cpu().numpy()
            fake_prob = probs[:, 1]
            pred = pred_out["pred"].detach().cpu().numpy()
            routes = pred_out["routes"].detach().cpu().numpy()
            conf = pred_out["conf"].detach().cpu().numpy()
            decisions = pred_out["decision"]

            labels = batch["label"].detach().cpu().numpy()
            resolutions = batch["resolution"].detach().cpu().numpy()

            for i in range(len(fake_prob)):
                path = str(batch["path"][i])
                rows.append({
                    "image_path": path,
                    "video_id": path_to_video.get(path, path),
                    "method": path_to_method.get(path, "unknown_method"),
                    "label": int(labels[i]),
                    "fake_prob": float(fake_prob[i]),
                    "pred_label": int(pred[i]),
                    "confidence": float(conf[i]),
                    "decision": str(decisions[i]),
                    "route": int(routes[i]),
                    "resolution": int(resolutions[i]),
                })

            now = time.time()
            if step == 1 or step == len(loader) or step % UI_UPDATE_EVERY_STEPS == 0 or now - last_update >= UI_UPDATE_EVERY_SECONDS:
                progress.update(
                    phase=str(stage_name),
                    current=step,
                    total=len(loader),
                    start_time=start_time,
                    extra=f"rows={len(rows)}",
                )
                last_update = now

    elapsed = time.time() - start_time
    return pd.DataFrame(rows), elapsed


def aggregate_video(frame_df):
    agg = {
        "label": ("label", "first"),
        "fake_prob": ("fake_prob", "mean"),
        "confidence": ("confidence", "mean"),
        "method": ("method", "first"),
        "num_frames": ("fake_prob", "size"),
    }
    return frame_df.groupby("video_id").agg(**agg).reset_index()


def save_eval_outputs(frame_df, elapsed, out_dir, device, batch_size, resolutions):
    import pandas as pd

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    video_df = aggregate_video(frame_df)

    frame_metrics = compute_metrics(
        frame_df["label"],
        frame_df["fake_prob"],
        threshold=DECISION_THRESHOLD,
        confidence=frame_df["confidence"],
        uncertain_threshold=UNCERTAIN_THRESHOLD,
    )

    video_metrics = compute_metrics(
        video_df["label"],
        video_df["fake_prob"],
        threshold=DECISION_THRESHOLD,
        confidence=video_df["confidence"],
        uncertain_threshold=UNCERTAIN_THRESHOLD,
    )

    metrics_df = pd.DataFrame([
        {"Level": "Frame", **frame_metrics},
        {"Level": "Video", **video_metrics},
    ])

    by_method_rows = []
    for method, g in frame_df.groupby("method"):
        row = compute_metrics(
            g["label"],
            g["fake_prob"],
            threshold=DECISION_THRESHOLD,
            confidence=g["confidence"],
            uncertain_threshold=UNCERTAIN_THRESHOLD,
        )
        row["method"] = method
        row["num_images"] = len(g)
        row["mean_fake_prob"] = float(g["fake_prob"].mean())
        row["mean_confidence"] = float(g["confidence"].mean())
        by_method_rows.append(row)

    by_method_df = pd.DataFrame(by_method_rows)
    if len(by_method_df) > 0 and "Fake Detection Rate / Recall(1)" in by_method_df.columns:
        by_method_df = by_method_df.sort_values("Fake Detection Rate / Recall(1)", ascending=True)

    inference_df = pd.DataFrame([{
        "frames_scored": len(frame_df),
        "videos_scored": video_df["video_id"].nunique(),
        "total_seconds": elapsed,
        "time_per_frame_ms": elapsed / max(len(frame_df), 1) * 1000.0,
        "fps": len(frame_df) / max(elapsed, 1e-8),
        "device": str(device),
        "batch_size": batch_size,
        "image_size": IMAGE_SIZE,
        "resolutions": ",".join(map(str, resolutions)),
    }])

    frame_df.to_csv(out_dir / "test_frame_predictions.csv", index=False)
    video_df.to_csv(out_dir / "test_video_predictions.csv", index=False)
    metrics_df.to_csv(out_dir / "test_metrics.csv", index=False)
    by_method_df.to_csv(out_dir / "test_by_method_metrics.csv", index=False)
    inference_df.to_csv(out_dir / "test_inference_time.csv", index=False)

    with open(out_dir / "test_metrics.json", "w") as f:
        json.dump(frame_metrics, f, indent=2)

    return metrics_df, by_method_df, inference_df


# ============================================================
# MAIN RESUME + EVAL
# ============================================================

print_header("AdaGate self-contained resume setup")
check_gpu_processes()

prepare_repo()
seed_everything(SEED)

import pandas as pd
import torch
from torch.cuda.amp import GradScaler
from deepfake_lowres.models.gated_dual_branch import GatedDualBranchDeepfakeNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

safe_df = make_safe_df40_manifest()
train_df, val_df, test_df = build_splits(safe_df)

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_EVAL.mkdir(parents=True, exist_ok=True)
OUT_OOD.mkdir(parents=True, exist_ok=True)

train_loader = make_loader(
    train_df,
    mode="train_random",
    resolutions=TRAIN_RESOLUTIONS,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = make_loader(
    val_df,
    mode="eval_all",
    resolutions=EVAL_RESOLUTIONS,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = make_loader(
    test_df,
    mode="eval_all",
    resolutions=EVAL_RESOLUTIONS,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

model = GatedDualBranchDeepfakeNet(branch_channels=(12, 20, 32, 48)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler(enabled=(MIXED_PRECISION and device.type == "cuda"))

history = []
start_epoch = 1
best_metric = -1.0

if HISTORY_CSV.exists():
    hist_df = pd.read_csv(HISTORY_CSV)
    history = hist_df.to_dict("records")
    if len(hist_df) > 0 and "epoch" in hist_df.columns:
        last_epoch = int(hist_df["epoch"].max())
        start_epoch = last_epoch + 1
        print("Existing history found:", HISTORY_CSV)
        print("Last completed epoch:", last_epoch)
    else:
        print("History CSV exists but has no usable epoch column.")

if BEST_CKPT.exists():
    best_data = torch.load(BEST_CKPT, map_location=device)
    best_metric = float(best_data.get("best_metric", best_data.get("current_metric", -1.0)))
    print("Best checkpoint found:", BEST_CKPT)
    print("Best metric so far:", best_metric)

if LATEST_CKPT.exists():
    print("Loading latest checkpoint:", LATEST_CKPT)
    ckpt = torch.load(LATEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)

    if "optimizer" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
            print("Loaded optimizer state.")
        except Exception as e:
            print("Could not load optimizer state. Continuing with fresh optimizer:", repr(e))

    if "epoch" in ckpt:
        start_epoch = max(start_epoch, int(ckpt["epoch"]) + 1)

    if "best_metric" in ckpt:
        best_metric = max(best_metric, float(ckpt["best_metric"]))

else:
    print("No latest checkpoint found. Starting from scratch.")

progress = LiveProgress("Official AdaGate-DF on DF40")

if RUN_TRAIN_RESUME and start_epoch <= EPOCHS:
    print(f"Resuming training from epoch {start_epoch} to epoch {EPOCHS}.")

    for epoch in range(start_epoch, EPOCHS + 1):
        train_stats = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            progress=progress,
            epoch_idx=epoch,
        )

        val_pred, val_elapsed = predict(
            model=model,
            loader=val_loader,
            device=device,
            progress=progress,
            stage_name=f"Val epoch {epoch}",
            source_df=val_df,
        )

        val_metrics = compute_metrics(
            val_pred["label"],
            val_pred["fake_prob"],
            threshold=DECISION_THRESHOLD,
            confidence=val_pred["confidence"],
            uncertain_threshold=UNCERTAIN_THRESHOLD,
        )

        metric = val_metrics["AUCROC"]
        if isinstance(metric, float) and math.isnan(metric):
            metric = val_metrics["Accuracy"]

        row = {
            "epoch": epoch,
            **train_stats,
            "val_metric_for_selection": metric,
            "val_accuracy": val_metrics["Accuracy"],
            "val_aucroc": val_metrics["AUCROC"],
            "val_auprc": val_metrics["AUPRC"],
            "val_eer": val_metrics["EER (Equal Error Rate)"],
            "val_recall_real": val_metrics["Recall (0)"],
            "val_recall_fake": val_metrics["Recall (1)"],
            "val_uncertain_rate": val_metrics["Uncertain Rate (%)"],
        }

        history = [r for r in history if int(r["epoch"]) != epoch]
        history.append(row)
        history = sorted(history, key=lambda r: int(r["epoch"]))

        pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

        ckpt = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "config": {
                "model": "official_adagate_gated_dual_branch",
                "dataset": "DF40",
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "image_size": IMAGE_SIZE,
                "train_resolutions": TRAIN_RESOLUTIONS,
                "eval_resolutions": EVAL_RESOLUTIONS,
                "frames_per_video": FRAMES_PER_VIDEO,
                "balance_mode": BALANCE_MODE,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "mixed_precision": MIXED_PRECISION,
                "decision_threshold": DECISION_THRESHOLD,
                "uncertain_threshold": UNCERTAIN_THRESHOLD,
                "seed": SEED,
            },
            "best_metric": best_metric,
            "current_metric": metric,
        }

        torch.save(ckpt, LATEST_CKPT)

        if metric > best_metric:
            best_metric = metric
            ckpt["best_metric"] = best_metric
            torch.save(ckpt, BEST_CKPT)

        print_header(f"Finished epoch {epoch}/{EPOCHS}")
        print("Train loss:", train_stats["train_loss"])
        print("Train accuracy:", train_stats["train_accuracy"])
        print("Val accuracy:", val_metrics["Accuracy"])
        print("Val AUCROC:", val_metrics["AUCROC"])
        print("Val AUPRC:", val_metrics["AUPRC"])
        print("Val EER:", val_metrics["EER (Equal Error Rate)"])
        print("Best metric:", best_metric)

else:
    print(f"Training already complete or resume disabled. start_epoch={start_epoch}, EPOCHS={EPOCHS}")

print_header("Loading best checkpoint for final evaluation")
if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    print("Loaded:", BEST_CKPT)
else:
    print("Best checkpoint missing. Using current model.")

if RUN_INDOMAIN_EVAL:
    print_header("Running in-domain DF40 final evaluation")

    frame_df, elapsed = predict(
        model=model,
        loader=test_loader,
        device=device,
        progress=progress,
        stage_name="Test",
        source_df=test_df,
    )

    metrics_df, by_method_df, inference_df = save_eval_outputs(
        frame_df=frame_df,
        elapsed=elapsed,
        out_dir=OUT_EVAL,
        device=device,
        batch_size=BATCH_SIZE,
        resolutions=EVAL_RESOLUTIONS,
    )

    print("Saved in-domain outputs to:", OUT_EVAL)
    display(metrics_df)
    display(inference_df)

if RUN_OOD_EVAL:
    print_header("Running OOD Fake_Test final evaluation")

    ood_df = build_ood_manifest()

    if len(ood_df) == 0:
        print("OOD manifest is empty. Skipping OOD evaluation.")
    else:
        ood_loader = make_loader(
            ood_df,
            mode="eval_all",
            resolutions=EVAL_RESOLUTIONS,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )

        ood_frame_df, ood_elapsed = predict(
            model=model,
            loader=ood_loader,
            device=device,
            progress=progress,
            stage_name="OOD",
            source_df=ood_df,
        )

        ood_metrics_df, ood_by_method_df, ood_inference_df = save_eval_outputs(
            frame_df=ood_frame_df,
            elapsed=ood_elapsed,
            out_dir=OUT_OOD,
            device=device,
            batch_size=BATCH_SIZE,
            resolutions=EVAL_RESOLUTIONS,
        )

        print("Saved OOD outputs to:", OUT_OOD)
        display(ood_metrics_df)
        display(ood_inference_df)
        display(ood_by_method_df.head(20))

print_header("Final file check")

for f in [
    HISTORY_CSV,
    LATEST_CKPT,
    BEST_CKPT,
    OUT_EVAL / "test_metrics.csv",
    OUT_EVAL / "test_frame_predictions.csv",
    OUT_EVAL / "test_video_predictions.csv",
    OUT_EVAL / "test_inference_time.csv",
    OUT_OOD / "test_metrics.csv",
    OUT_OOD / "test_frame_predictions.csv",
    OUT_OOD / "test_video_predictions.csv",
    OUT_OOD / "test_by_method_metrics.csv",
    OUT_OOD / "test_inference_time.csv",
]:
    print(("FOUND   " if f.exists() else "MISSING "), f)


AdaGate self-contained resume setup

GPU process check
292, /opt/conda/bin/python, 1312
FOUND AdaGate repo: /home/jovyan/Deepfake_Low Resource_Low Resolution/AdaGate-DF
FOUND DF40 clean manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/df40_manifest_all_clean.csv
FOUND DF40 Fake_Test folder: /home/jovyan/Deepfake_Low Resource_Low Resolution/df40/Fake_Test
AdaGate package ready: /home/jovyan/Deepfake_Low Resource_Low Resolution/AdaGate-DF/src/deepfake_lowres
Device: cuda


/tmp/ipykernel_4842/1765755115.py:236: DtypeWarning: Columns (0: unclear_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DF40_MANIFEST)


Saved safe manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40/dataset_summary/df40_manifest_adagate_safe.csv
Rows: 292907 -> 292907
label      0       1
split               
test    5999   22717
train  48132  187508
val     5997   22554

Split summary after sampling/limits:
train real=  48132 fake= 176341 total= 224473
val   real=   5997 fake=  22055 total=  28052
test  real=   5999 fake=  22004 total=  28003
Existing history found: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/training_period_results.csv
Last completed epoch: 8
Best checkpoint found: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/best_gated_dual_branch_df40.pt
Best metric so far: 0.8067975800036344
Loading latest checkpoint: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/latest_gated_dual_branch_df40.pt
Loaded optimizer state.


Resuming training from epoch 9 to epoch 10.

Finished epoch 9/10
Train loss: 0.40592597187058244
Train accuracy: 0.8504125383134935
Val accuracy: 0.8067873948381578
Val AUCROC: 0.805143930689746
Val AUPRC: 0.9371058173011482
Val EER: 0.26696787901242996
Best metric: 0.8067975800036344

Finished epoch 10/10
Train loss: 0.4009129954715084
Train accuracy: 0.8551527193670254
Val accuracy: 0.8067160986738914
Val AUCROC: 0.8107647415485872
Val AUPRC: 0.9386237979526518
Val EER: 0.26060307415099526
Best metric: 0.8107647415485872

Loading best checkpoint for final evaluation
Loaded: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/best_gated_dual_branch_df40.pt

Running in-domain DF40 final evaluation
Saved in-domain outputs to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain_eval


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),TPR@1%FPR,TPR@0.1%FPR,Precision (0),...,Precision (1),Recall (1),F1 (1),Real Support,Fake Support,Uncertain Rate (%),Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate
0,Frame,0.5,0.804235,0.563581,0.943065,0.816682,0.263695,0.280903,0.164697,0.716681,...,0.808131,0.984639,0.887696,5999,22004,10.648859,1193,26810,0.984639,0.015361
1,Video,0.5,0.825688,0.559704,0.965631,0.869235,0.220328,0.395202,0.328283,0.800000,...,0.826498,0.992424,0.901893,189,792,5.912334,30,951,0.992424,0.007576


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size,resolutions
0,28003,981,2138.318231,76.360327,13.095806,cuda,16,224,224



Running OOD Fake_Test final evaluation
Saved OOD manifest: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_df40_ood_fake_test_all_models/df40_fake_test_manifest_adagate_official.csv
OOD valid images: 175667
OOD bad images: 0
Saved OOD outputs to: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_ood_fake_test


,Level,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),TPR@1%FPR,TPR@0.1%FPR,Precision (0),...,Precision (1),Recall (1),F1 (1),Real Support,Fake Support,Uncertain Rate (%),Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate
0,Frame,0.5,0.903516,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,1.0,0.903516,0.949313,0,175667,7.244958,16949,158718,0.903516,0.096484
1,Video,0.5,0.903516,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,1.0,0.903516,0.949313,0,175667,7.244958,16949,158718,0.903516,0.096484


,frames_scored,videos_scored,total_seconds,time_per_frame_ms,fps,device,batch_size,image_size,resolutions
0,175667,175667,5286.057902,30.091354,33.232137,cuda,16,224,224


,Decision Threshold,Accuracy,Balanced Accuracy,AUPRC,AUCROC,EER (Equal Error Rate),TPR@1%FPR,TPR@0.1%FPR,Precision (0),Recall (0),...,Fake Support,Uncertain Rate (%),Predicted Real Count,Predicted Fake Count,Fake Detection Rate / Recall(1),Missed Fake Rate,method,num_images,mean_fake_prob,mean_confidence
1,0.5,0.552266,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,31885,23.663164,14276,17609,0.552266,0.447734,DiT,31885,0.499144,0.764515
6,0.5,0.933390,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,4669,18.055258,311,4358,0.933390,0.066610,deepfacelab,4669,0.778176,0.799767
2,0.5,0.943284,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,24790,9.249697,1406,23384,0.943284,0.056716,blendface,24790,0.810743,0.831498
3,0.5,0.983849,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,46190,2.626110,746,45444,0.983849,0.016151,cdf,46190,0.924773,0.930315
7,0.5,0.990049,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,10552,3.610690,105,10447,0.990049,0.009951,ff,10552,0.888368,0.890748
4,0.5,0.995653,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,23696,1.548785,103,23593,0.995653,0.004347,danet,23696,0.908239,0.909365
0,0.5,0.999000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,2000,4.100000,2,1998,0.999000,0.001000,CollabDiff,2000,0.860802,0.860857
5,0.5,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,...,31885,0.009409,0,31885,1.000000,0.000000,ddim,31885,0.964601,0.964601



Final file check
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/training_period_results.csv
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/latest_gated_dual_branch_df40.pt
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain/best_gated_dual_branch_df40.pt
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain_eval/test_metrics.csv
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain_eval/test_frame_predictions.csv
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain_eval/test_video_predictions.csv
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_indomain_eval/test_inference_time.csv
FOUND    /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_adagate_df40_ood_fake_test/test_metrics.csv
FOUND    /home/jovyan/Deepfake_Low Resour

In [ ]:
results = run_pipeline()

In [14]:
!pip install ipywidgets deepfake_lowres

ERROR: Could not find a version that satisfies the requirement deepfake_lowres (from versions: none)
ERROR: No matching distribution found for deepfake_lowres


In [1]:
 print('test')

test
